In [2]:
import pandas as pd

In [68]:
df = pd.read_csv('assets/shot_logs.csv')

### Clean dataset

In [69]:
COLUMNS_TO_DROP = [
    'GAME_ID', 'MATCHUP', 'W', 'FINAL_MARGIN', 'PTS', 'SHOT_RESULT', 'CLOSEST_DEFENDER', 'player_name'
]
df = df.drop(columns=COLUMNS_TO_DROP)

df['LOCATION'] = df['LOCATION'].map({'A': 0, 'H': 1})
df['PTS_TYPE'] = df['PTS_TYPE'].map({2: 0, 3: 1})
df['GAME_CLOCK'] = pd.to_timedelta('00:' + df['GAME_CLOCK']).dt.total_seconds()
df['SHOT_CLOCK'] = df['SHOT_CLOCK'].fillna(24)
df['TOUCH_TIME'] = df['TOUCH_TIME'].clip(lower=0).fillna(0)

In [70]:
Y = df['FGM'].values
df = df.drop(columns=['FGM'])
X = df.values

### Split train/dev/test datasets

In [ ]:
from sklearn.model_selection import train_test_split

# 80%/10%/10% split for train/dev/test datasets
X_trainval, X_test, Y_trainval, Y_test = train_test_split(X, Y, 
    test_size=0.1, 
    random_state=42,
    stratify=Y,
)
X_train, X_dev, Y_train, Y_dev = train_test_split(X_trainval, Y_trainval,
    test_size=0.1111,  # 0.1111 * 0.9 ≈ 0.1, so that dev set is 10% of the total
    random_state=42,
    stratify=Y_trainval,
)

### Target encode high-cardinality (only train dataset)

Deal with `player_id` and `CLOSEST_DEFENDER_PLAYER_ID` replacing each id with that players's mean `FGM`. (Avoid embedding layers for now)

### Standardize numeric features (only train dataset)